In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from PIL import Image, UnidentifiedImageError

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_DIR = #file name

IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 80
NUM_CLASSES = 6

############################################
# Safe Loader
############################################

def safe_loader(path):
    try:
        with Image.open(path) as img:
            return img.convert("RGB")
    except (UnidentifiedImageError, OSError):
        print("Skipping corrupted:", path)
        return Image.new("RGB",(IMG_SIZE,IMG_SIZE))

############################################
# Transforms
############################################

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

############################################
# Dataset
############################################

train_ds = datasets.ImageFolder(
    os.path.join(DATA_DIR,"train"),
    loader=safe_loader,
    transform=train_transform
)

val_ds = datasets.ImageFolder(
    os.path.join(DATA_DIR,"val"),
    loader=safe_loader,
    transform=val_transform
)

train_loader = DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True)
val_loader = DataLoader(val_ds,batch_size=BATCH_SIZE)

############################################
# Residual Block
############################################

class ResidualBlock(nn.Module):
    def __init__(self,channels):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(channels,channels,3,1,1),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(channels,channels,3,1,1),
            nn.BatchNorm2d(channels)
        )

        self.relu = nn.ReLU()

    def forward(self,x):
        return self.relu(x + self.conv(x))

############################################
# Attention Block (CBAM simplified)
############################################

class ChannelAttention(nn.Module):

    def __init__(self,channels):
        super().__init__()

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(
            nn.Linear(channels,channels//4),
            nn.ReLU(),
            nn.Linear(channels//4,channels),
            nn.Sigmoid()
        )

    def forward(self,x):

        b,c,_,_ = x.size()

        y = self.pool(x).view(b,c)
        y = self.fc(y).view(b,c,1,1)

        return x * y

############################################
# SolarFaultNet
############################################

class SolarFaultNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.layer1 = nn.Sequential(
            nn.Conv2d(3,32,3,1,1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.layer2 = nn.Sequential(
            nn.Conv2d(32,64,3,1,1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.res1 = ResidualBlock(64)

        self.layer3 = nn.Sequential(
            nn.Conv2d(64,128,3,1,1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.res2 = ResidualBlock(128)

        self.att = ChannelAttention(128)

        self.layer4 = nn.Sequential(
            nn.Conv2d(128,256,3,1,1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128,NUM_CLASSES)
        )

    def forward(self,x):

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.res1(x)

        x = self.layer3(x)
        x = self.res2(x)

        x = self.att(x)

        x = self.layer4(x)

        x = self.pool(x).flatten(1)

        return self.fc(x)

model = SolarFaultNet().to(DEVICE)

############################################
# Training setup
############################################

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,EPOCHS)

best_acc = 0

############################################
# Training loop with checkpoints
############################################

for epoch in range(EPOCHS):

    model.train()

    total = 0
    correct = 0
    loss_sum = 0

    for imgs,labels in train_loader:

        imgs = imgs.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        out = model(imgs)

        loss = criterion(out,labels)

        loss.backward()

        optimizer.step()

        loss_sum += loss.item()

        pred = out.argmax(1)

        correct += (pred==labels).sum().item()
        total += labels.size(0)

    train_acc = 100*correct/total

    ############################################
    # Validation
    ############################################

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for imgs,labels in val_loader:

            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)

            out = model(imgs)

            pred = out.argmax(1)

            correct += (pred==labels).sum().item()
            total += labels.size(0)

    val_acc = 100*correct/total

    scheduler.step()

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Acc {train_acc:.2f}% | Val Acc {val_acc:.2f}%")

    ############################################
    # Save best model
    ############################################

    if val_acc > best_acc:

        best_acc = val_acc

        torch.save(model.state_dict(),"solar_fault_best.pth")

        print("Model checkpoint saved")

In [ ]:
import random
import torch
import matplotlib.pyplot as plt

model.eval()

class_names = val_ds.classes

samples_per_class = 2   

for class_idx, class_name in enumerate(class_names):

    # collect indices belonging to this class
    indices = [i for i, (_, label) in enumerate(val_ds) if label == class_idx]

    # randomly pick images
    random_samples = random.sample(indices, min(samples_per_class, len(indices)))

    for idx in random_samples:

        img, label = val_ds[idx]

        input_tensor = img.unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            output = model(input_tensor)
            pred = torch.argmax(output, 1).item()

        pred_label = class_names[pred]

        # convert tensor to image
        img_display = img.permute(1,2,0)

        # unnormalize
        img_display = img_display * torch.tensor([0.229,0.224,0.225]) + torch.tensor([0.485,0.456,0.406])
        img_display = img_display.clamp(0,1)

        plt.imshow(img_display)
        plt.title(f"True: {class_name} | Predicted: {pred_label}")
        plt.axis("off")
        plt.show()


In [14]:
import torch

model = SolarFaultNet().to(DEVICE)

model.load_state_dict(torch.load("solar_fault_best.pth", map_location=DEVICE))

model.eval()

print("Checkpoint loaded successfully")

In [15]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

all_preds = []
all_labels = []

with torch.no_grad():

    for imgs, labels in val_loader:

        imgs = imgs.to(DEVICE)

        outputs = model(imgs)

        preds = torch.argmax(outputs,1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

class_names = val_ds.classes

In [8]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Solar Panel Fault Classification")

plt.show()

In [9]:
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(8,6))

sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="Greens",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Normalized Confusion Matrix")

plt.show()

In [10]:
print("\nClassification Report:\n")

print(classification_report(
    all_labels,
    all_preds,
    target_names=class_names
))

In [ ]:
print("\nPer-Class Accuracy:\n")

for i, class_name in enumerate(class_names):

    class_total = cm[i].sum()
    class_correct = cm[i][i]

    acc = class_correct / class_total

    print(f"{class_name}: {acc*100:.2f}%")